# Run Autoformalism in Colab

This notebook keeps benchmark data and experiment checkpoints in Google Drive. Change the repository URL, benchmark root, model identifiers, and benchmark selection before running.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
import os
import subprocess
from pathlib import Path

REPOSITORY_URL = 'https://github.com/YOUR_ORG/autoformalism.git'
REPOSITORY = Path('/content/autoformalism')
DATA_ROOT = Path('/content/drive/MyDrive/autoformalism/data_raw')
OUTPUT_ROOT = Path('/content/drive/MyDrive/autoformalism/runs')

if (REPOSITORY / '.git').is_dir():
    subprocess.run(['git', '-C', str(REPOSITORY), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', REPOSITORY_URL, str(REPOSITORY)], check=True)

subprocess.run(['python', '-m', 'pip', 'install', '-e', str(REPOSITORY)], check=True)

In [ ]:
from google.colab import userdata

api_key = userdata.get('OPENAI_API_KEY')
if not api_key:
    raise RuntimeError('Add OPENAI_API_KEY under Colab > Secrets before continuing.')
os.environ['OPENAI_API_KEY'] = api_key
os.environ['AUTOFORMALISM_DATA_ROOT'] = str(DATA_ROOT)

## Validate benchmark paths and metadata

In [ ]:
BENCHMARK_ID = 'original_b1'
TIER = 'easy'
SEED = 0
PROPOSER_MODEL = 'openai:YOUR_PROPOSER_MODEL'
JUDGE_MODEL = 'openai:YOUR_JUDGE_MODEL'
ITERATIONS = 5
BEAM_SIZE = 2

if not DATA_ROOT.is_dir():
    raise FileNotFoundError(f'Benchmark root does not exist: {DATA_ROOT}')

common = [
    '--data-root', str(DATA_ROOT),
    '--benchmark-id', BENCHMARK_ID,
    '--tier', TIER,
    '--seed', str(SEED),
    '--proposer-model', PROPOSER_MODEL,
    '--judge-model', JUDGE_MODEL,
    '--iteration-budget', str(ITERATIONS),
    '--beam-size', str(BEAM_SIZE),
    '--output-root', str(OUTPUT_ROOT),
]
subprocess.run(
    ['python', str(REPOSITORY / 'scripts/run_experiment.py'), *common, '--dry-run'],
    check=True,
)

## Run one benchmark

In [ ]:
subprocess.run(
    ['python', str(REPOSITORY / 'scripts/run_experiment.py'), *common],
    check=True,
)

## Resume after an interruption

Run this cell instead of the previous cell when the experiment directory already contains checkpoints.

In [ ]:
subprocess.run(
    ['python', str(REPOSITORY / 'scripts/resume_experiment.py'), *common],
    check=True,
)

## Summarize saved results

In [ ]:
subprocess.run(
    ['python', str(REPOSITORY / 'scripts/summarize_results.py'), str(OUTPUT_ROOT)],
    check=True,
)